# Prueba previa: extracción en JSON

Este notebook no es la transformación de género. Esa solución está en `transformacion_genero_musical.ipynb`. Aquí se probó un primer paso con `blues.00003.wav`: que un modelo devolviera datos de la canción en JSON, por si ese JSON orientaba después la versión nueva.

El JSON describe la pista que ya existe. No genera la canción transformada, no mantiene una conversación y no consulta una base de conocimiento. Por eso esta prueba queda fuera del flujo final.

## Librerías de esta prueba

| Librería | Uso |
|---|---|
| OpenAI (`gpt-4o-transcribe` y `gpt-4o`) | Transcribe el audio y rellena los JSON. |
| LangChain (`ChatOpenAI`) | Hace la llamada a `gpt-4o`. |
| librosa y numpy | Miden duración, tempo, centroide espectral y cruces por cero. |
| pandas, jq, pypdf, chromadb y langchain-classic | Se instalaron en la sesión de Colab y no intervienen en la extracción. |


## 1. Entorno

La primera celda es la instalación de aquella sesión. La extracción usa `openai`, `langchain-openai`, `librosa` y `numpy`. El resto de paquetes no se llama en el código de abajo.

La clave de OpenAI se lee de `OPENAI_API_KEY` en el entorno o en los secretos de Colab.


In [ ]:
# las versiones están fijadas para replicabilidad el día de la clase, pero no es necesario:
!pip install langchain==1.0.0
!pip install langchain-classic
!pip install langchain-openai
!pip install langchain-community
!pip install jq
!pip install pypdf
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.4.4
    Uninstalling langgraph-sdk-0.4.4:
      Successfully uninstalled langgraph-sdk-0.4.4
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.1.0
    Uninstalling langgraph-prebuilt-1.1.0:
      Successfully uninstalled langgraph-prebuilt-1.1.0
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.11
    Uninstalling langgraph-1.2.11:
      Successfully uninstalled langgraph-1.2.11
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.18
    Uninstalling langchain-1.3.18:
      Successfully uninstalled langchain-1.3.18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/

In [ ]:
import os
from langchain_openai import ChatOpenAI


In [ ]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = (userdata.get("OPENAI_API_KEY") or "").strip()
    except Exception:
        OPENAI_API_KEY = ""
if not OPENAI_API_KEY:
    raise RuntimeError(
        "Define OPENAI_API_KEY en el entorno o en los secretos de Colab. No la pegues en el notebook."
    )
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


## 2. Transcripción

El audio se sube en Colab y `gpt-4o-transcribe` devuelve el texto cantado. En esta pista el resultado fue: «I had a friend one time, least I thought I did.»


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving blues.00003.wav to blues.00003.wav


In [ ]:
cancion = "blues.00003.wav"
cancion1 = open(cancion, "rb")

In [ ]:
from openai import OpenAI
openai_client = OpenAI()

transcripcion = openai_client.audio.transcriptions.create(
    model='gpt-4o-transcribe',
    file=cancion1
)

texto = transcripcion.text
print(texto)

I had a friend one time, least I thought I did.


## 3. JSON de la letra

`gpt-4o` devuelve solo un JSON con tema, emoción, energía y descripción. Es una ficha de la letra, no una versión nueva de la canción.


In [ ]:
chat_llm = ChatOpenAI(model_name='gpt-4o')
import json

analisis = chat_llm.invoke(
    f"""
    Analiza esta letra:

    {texto}

    Devuelve únicamente JSON válido con:

    {{
      "tema":"",
      "emocion":"",
      "energia":"",
      "descripcion":""
    }}
    """
)

print(analisis.content)


```json
{
  "tema": "amistad y decepción",
  "emocion": "tristeza",
  "energia": "baja",
  "descripcion": "La letra refleja la experiencia de sentirse traicionado o decepcionado por alguien a quien se consideraba un amigo. La emoción predominante es la tristeza debido a la pérdida de confianza y el malestar emocional causado por la ruptura de una relación que se pensaba genuina."
}
```


In [ ]:
descripcion = analisis.content

## 4. JSON de arreglo a partir de esa ficha

Con la descripción anterior se pidió otro JSON, ahora de instrumentación, ritmo, voz y producción en jazz, más un prompt para una IA musical. Sigue siendo un encargo escrito. La transformación que se entrega no partió de este JSON.


In [ ]:
prompt = f"""
Descripción de la canción:

{descripcion}

Género objetivo: Jazz

Genera:

- Instrumentación
- Ritmo
- Voz
- Producción
- Prompt final para IA musical

Devuelve JSON.
"""

response = chat_llm.invoke([("human", prompt)])
print(response.content)

```json
{
  "instrumentacion": {
    "piano": "Un piano acústico suave y melancólico que proporciona la base armónica, resaltando las emociones de tristeza y decepción.",
    "contrabajo": "Una línea de contrabajo que acompaña de manera sutil, proporcionando profundidad y soporte al piano.",
    "saxofón": "Un saxofón tenor que establece el tono melancólico, con frases largas y sostenidas que evocan sentimientos de tristeza.",
    "percusión": "Un kit de batería clásico de jazz, tocado con escobillas para mantener un ritmo suave y delicado, enfatizando el ambiente íntimo."
  },
  "ritmo": {
    "tempo": "Lento",
    "compás": "4/4",
    "descripcion": "Un ritmo lento y pausado que permite el espacio necesario para la reflexión y enfatiza la sensación de tristeza."
  },
  "voz": {
    "tipo": "Voz masculina suave y algo ronca",
    "estilo": "Interpretación vocal cargada de emoción, con un tono íntimo y sincero que complementa la letra introspectiva.",
    "armonías": "Voces secundarias

## 5. JSON a partir de medidas del audio

Librosa mide la señal. No reconoce instrumentos. Esas cifras se pasaron a `gpt-4o` con el género jazz. El prompt resultante describe una pieza nueva de unos 63 BPM; no conserva la melodía de `blues.00003.wav`. Por eso no se usó en la transformación.


In [ ]:
# Instalar dependencias
!pip install -q openai langchain-openai librosa

from google.colab import files
from langchain_openai import ChatOpenAI
import librosa
import numpy as np
import json

# Subir canción
uploaded = files.upload()

# Obtener nombre del fichero
cancion = list(uploaded.keys())[0]

print("Canción:", cancion)

# Cargar audio
audio, sr = librosa.load(cancion, sr=None)

# Extraer características
tempo, _ = librosa.beat.beat_track(y=audio, sr=sr)

features = {
    "nombre_archivo": cancion,
    "duracion_segundos": round(
        librosa.get_duration(y=audio, sr=sr), 2
    ),
    "tempo_bpm": round(float(np.atleast_1d(tempo)[0]), 2),
    "spectral_centroid": round(
        float(np.mean(
            librosa.feature.spectral_centroid(
                y=audio,
                sr=sr
            )
        )),
        2
    ),
    "zero_crossing_rate": round(
        float(np.mean(
            librosa.feature.zero_crossing_rate(audio)
        )),
        5
    )
}

print(features)

# Género destino
genero_objetivo = "Jazz"

# Modelo
chat_llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7
)

prompt = f"""
Eres un productor musical experto.

Características extraídas del audio:

{json.dumps(features, indent=2)}

Género objetivo:
{genero_objetivo}

Analiza la canción y genera un JSON con:

{{
  "energia":"",
  "emocion":"",
  "instrumentacion":[
  ],
  "ritmo":"",
  "voz":"",
  "produccion":"",
  "prompt_final_para_ia_musical":""
}}

Devuelve únicamente JSON válido.
"""

response = chat_llm.invoke(prompt)

print(response.content)



Saving blues.00003.wav to blues.00003 (2).wav
Canción: blues.00003 (2).wav
{'nombre_archivo': 'blues.00003 (2).wav', 'duracion_segundos': 30.01, 'tempo_bpm': 63.02, 'spectral_centroid': 1070.15, 'zero_crossing_rate': 0.03331}


```json
{
  "energia": "baja",
  "emocion": "relajada",
  "instrumentacion": [
    "piano",
    "contrabajo",
    "batería",
    "saxofón"
  ],
  "ritmo": "lento",
  "voz": "sin voz",
  "produccion": "suave y acústica",
  "prompt_final_para_ia_musical": "Crea una pieza de jazz relajada y acústica que combine piano, contrabajo, batería y saxofón con un tempo lento de aproximadamente 63 BPM, sin voz."
}
```


## 6. Qué se concluye de esta prueba

Los dos JSON, el de la letra y el de las medidas, documentan la canción de partida. No hacen falta para generar la versión en el género objetivo. Esa generación usa el prompt de `transformacion_genero_musical.ipynb`.
